# Late Delivery Prediction — Olist E-Commerce Dataset

This project aims to predict whether an order will be delivered late using transactional and logistics-related features.

The goal is not only to build a model, but to understand the trade-offs between recall and precision in a real-world decision setting.

## Problem Definition

Late deliveries negatively impact customer satisfaction and operational efficiency.

This project focuses on:
- Predicting whether an order will be delivered late
- Understanding key drivers of delivery delays
- Evaluating trade-offs between catching delays vs avoiding false alarms

In [1]:
import pandas as pd
import numpy as np

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/enzoschitini/brazilian-e-commerce-public-dataset-by-olist/Brazilian E-Commerce Public Dataset by Olist.csv


## Data Preparation

The dataset contains multiple rows per order due to multiple items.

To ensure consistency:
- Data is aggregated to order level
- Each row represents one order

In [2]:
#load dataset
df = pd.read_csv("/kaggle/input/datasets/enzoschitini/brazilian-e-commerce-public-dataset-by-olist/Brazilian E-Commerce Public Dataset by Olist.csv")

In [4]:
df.shape


(113390, 39)

In [6]:
df.head()

,Unnamed: 0,order_id,order_item_id,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_id,product_category_name,...,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,day_of_purchase,month_of_purchase,year_of_purchase,month/year_of_purchase,order_status,order_unique_id
0,0,00010242fe8c5a6d1ba2dd792cb16214,1,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,...,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,Wednesday,September,2017,September-2017,delivered,00010242fe8c5a6d1ba2dd792cb16214-1
1,1,130898c0987d1801452a8ed92a670612,1,e6eecc5a77de221464d1c4eaff0a9b64,0fb8e3eab2d3e79d92bb3fffbb97f188,75800,jatai,GO,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,...,2017-06-29 02:44:11,2017-07-05 12:00:33,2017-07-13 20:39:29,2017-07-26 00:00:00,Wednesday,June,2017,June-2017,delivered,130898c0987d1801452a8ed92a670612-1
2,2,532ed5e14e24ae1f0d735b91524b98b9,1,4ef55bf80f711b372afebcb7c715344a,3419052c8c6b45daf79c1e426f9e9bcb,30720,belo horizonte,MG,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,...,2018-05-18 12:31:43,2018-05-23 14:05:00,2018-06-04 18:34:26,2018-06-07 00:00:00,Friday,May,2018,May-2018,delivered,532ed5e14e24ae1f0d735b91524b98b9-1
3,3,6f8c31653edb8c83e1a739408b5ff750,1,30407a72ad8b3f4df4d15369126b20c9,e7c828d22c0682c1565252deefbe334d,83070,sao jose dos pinhais,PR,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,...,2017-08-01 18:55:08,2017-08-02 19:07:36,2017-08-09 21:26:33,2017-08-25 00:00:00,Tuesday,August,2017,August-2017,delivered,6f8c31653edb8c83e1a739408b5ff750-1
4,4,7d19f4ef4d04461989632411b7e588b9,1,91a792fef70ecd8cc69d3c7feb3d12da,0bb98ba72dcc08e95f9d8cc434e9a2cc,36400,conselheiro lafaiete,MG,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,...,2017-08-10 22:05:11,2017-08-11 19:43:07,2017-08-24 20:04:21,2017-09-01 00:00:00,Thursday,August,2017,August-2017,delivered,7d19f4ef4d04461989632411b7e588b9-1


In [3]:

df.columns

Index(['Unnamed: 0', 'order_id', 'order_item_id', 'customer_id',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'product_id', 'product_category_name',
       'product_name_lenght', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm', 'seller_id', 'seller_city',
       'seller_state', 'seller_zip_code_prefix', 'payment_type',
       'payment_sequential', 'payment_installments', 'price', 'freight_value',
       'payment_value', 'shipping_limit_date', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'day_of_purchase', 'month_of_purchase', 'year_of_purchase',
       'month/year_of_purchase', 'order_status', 'order_unique_id'],
      dtype='object')

In [7]:
#drop kolom
df = df.drop(columns=["Unnamed: 0"])

In [8]:
df['order_id'].nunique(), len(df)

(95128, 113390)

In [9]:
#cek duplikasi order
df.groupby('order_id').size().value_counts().head()

1    83072
2     9128
3     1517
4      754
6      263
Name: count, dtype: int64

In [87]:
#Agregate
agg_df = df.groupby('order_id').agg({
    'customer_unique_id': 'first',
    'customer_state': 'first',
    'customer_city': 'first',
    
    'seller_id': 'first',  
    
    'seller_state': 'first',
    'seller_city': 'first',
    
    'price': 'sum',
    'freight_value': 'sum',
    'payment_value': 'sum',
    'payment_installments': 'max',
    
    'product_weight_g': 'sum',
    'product_length_cm': 'mean',
    'product_height_cm': 'mean',
    'product_width_cm': 'mean',
    
    'product_photos_qty': 'mean',
    
    'order_purchase_timestamp': 'first',
    'order_approved_at': 'first',
    'order_delivered_customer_date': 'first',
    'order_estimated_delivery_date': 'first',
    
    'order_status': 'first'
}).reset_index()

In [88]:
agg_df.shape

(95128, 21)

In [89]:
agg_df.head()

,order_id,customer_unique_id,customer_state,customer_city,seller_id,seller_state,seller_city,price,freight_value,payment_value,...,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_photos_qty,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,order_estimated_delivery_date,order_status
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,RJ,campos dos goytacazes,48436dade18ac8b2bce089ec2a041202,SP,volta redonda,58.90,13.29,72.19,...,650.0,28.0,9.0,14.0,4.0,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-20 23:43:48,2017-09-29 00:00:00,delivered
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,SP,santa fe do sul,dd7ddc04e1b6c2c614352b383efe2d36,SP,sao paulo,239.90,19.93,259.83,...,30000.0,50.0,30.0,40.0,2.0,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-12 16:04:24,2017-05-15 00:00:00,delivered
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,MG,para de minas,5b51032eddd242adc84c38acab88f23d,MG,borda da mata,199.00,17.87,216.87,...,3050.0,33.0,13.0,33.0,2.0,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-22 13:19:16,2018-02-05 00:00:00,delivered
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,SP,atibaia,9d7a1d34a5052409006425275ba1c2b4,SP,franca,12.99,12.79,25.78,...,200.0,16.0,10.0,15.0,1.0,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-14 13:32:39,2018-08-20 00:00:00,delivered
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,SP,varzea paulista,df560393f3a51e74553ab94004ba5c87,PR,loanda,199.90,18.14,218.04,...,3750.0,35.0,40.0,30.0,1.0,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-03-01 16:42:31,2017-03-17 00:00:00,delivered


## Target Definition

An order is considered late if:

order_delivered_customer_date > order_estimated_delivery_date

This creates a binary classification problem.

In [90]:
#buat target is late
agg_df['is_late'] = (
    agg_df['order_delivered_customer_date'] > agg_df['order_estimated_delivery_date']
).astype(int)

In [91]:
#distribusi target
agg_df['is_late'].value_counts(normalize=True)

is_late
0    0.918993
1    0.081007
Name: proportion, dtype: float64

In [92]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    agg_df[col] = pd.to_datetime(agg_df[col], errors='coerce')

In [93]:
agg_df[date_cols].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## Feature Engineering

We engineered features that reflect real-world logistics constraints:

- approval_time_hours → processing delay before shipment
- estimated_delivery_days → expected delivery window
- freight_ratio → proxy for shipping complexity
- delivery_pressure → tightness of delivery timeline
- is_peak_month → captures seasonality effects

These features aim to approximate operational bottlenecks.

In [101]:
#Feature engineering
agg_df['is_late'] = (
    agg_df['order_delivered_customer_date'] > agg_df['order_estimated_delivery_date']
).astype(int)

agg_df['approval_time_hours'] = (
    agg_df['order_approved_at'] - agg_df['order_purchase_timestamp']
).dt.total_seconds() / 3600

agg_df['freight_ratio'] = agg_df['freight_value'] / (agg_df['price'] + 1)

agg_df['purchase_hour'] = agg_df['order_purchase_timestamp'].dt.hour
agg_df['purchase_dayofweek'] = agg_df['order_purchase_timestamp'].dt.dayofweek
agg_df['purchase_month'] = agg_df['order_purchase_timestamp'].dt.month

agg_df['same_state'] = (
    agg_df['customer_state'] == agg_df['seller_state']
).astype(int)

agg_df['estimated_delivery_days'] = (
    agg_df['order_estimated_delivery_date'] - agg_df['order_purchase_timestamp']
).dt.days

In [102]:
agg_df[['is_late', 'estimated_delivery_days', 'approval_time_hours']].head()

,is_late,estimated_delivery_days,approval_time_hours
0,0,15,0.775833
1,0,18,0.201944
2,0,21,0.249722
3,0,11,0.161944
4,0,40,0.206111


In [103]:
seller_count = agg_df['seller_id'].value_counts().reset_index()
seller_count.columns = ['seller_id', 'seller_order_count']

agg_df = agg_df.merge(seller_count, on='seller_id', how='left')

In [106]:
agg_df = agg_df.drop(columns=['seller_order_count_x'], errors='ignore')
agg_df = agg_df.rename(columns={'seller_order_count_y': 'seller_order_count'})

In [107]:
agg_df[['seller_id', 'seller_order_count']].head()
agg_df['seller_order_count'].describe()

count    95128.000000
mean       372.874317
std        486.074931
min          1.000000
25%         51.000000
50%        148.000000
75%        465.000000
max       1819.000000
Name: seller_order_count, dtype: float64

In [96]:
item_count = df.groupby('order_id').size().reset_index(name='item_count')

In [97]:
agg_df = agg_df.merge(item_count, on='order_id', how='left')

In [98]:
agg_df[['order_id', 'item_count']].head()
agg_df['item_count'].describe()

count    95128.000000
mean         1.191973
std          0.730957
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         63.000000
Name: item_count, dtype: float64

In [99]:
seller_count = agg_df['seller_id'].value_counts().reset_index()
seller_count.columns = ['seller_id', 'seller_order_count']

agg_df = agg_df.merge(seller_count, on='seller_id', how='left')

In [119]:
agg_df['approval_to_estimated_days'] = (
    agg_df['order_estimated_delivery_date'] - agg_df['order_approved_at']
).dt.days

In [140]:
agg_df['is_peak_month'] = agg_df['purchase_month'].isin([11, 12]).astype(int)

In [151]:
agg_df['delivery_pressure'] = (
    agg_df['estimated_delivery_days'] / (agg_df['approval_time_hours'] + 1)
)

In [170]:
#Fitur seleksi
features = [
    'customer_state',
    'seller_state',
    'price',
    'freight_value',
    'payment_value',
    'payment_installments',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
    'product_photos_qty',
    'approval_time_hours',
    'freight_ratio',
    'purchase_hour',
    'purchase_dayofweek',
    'purchase_month',
    'same_state',
    'estimated_delivery_days',
    'seller_order_count',
    'item_count',
    'approval_to_estimated_days',
    'is_peak_month',
    'delivery_pressure'
]

model_df = agg_df[features + ['is_late']].copy()

In [171]:
#cek missing fitur
model_df.isnull().mean().sort_values(ascending=False)

customer_state                0.0
seller_state                  0.0
price                         0.0
freight_value                 0.0
payment_value                 0.0
payment_installments          0.0
product_weight_g              0.0
product_length_cm             0.0
product_height_cm             0.0
product_width_cm              0.0
product_photos_qty            0.0
approval_time_hours           0.0
freight_ratio                 0.0
purchase_hour                 0.0
purchase_dayofweek            0.0
purchase_month                0.0
same_state                    0.0
estimated_delivery_days       0.0
seller_order_count            0.0
item_count                    0.0
approval_to_estimated_days    0.0
is_peak_month                 0.0
delivery_pressure             0.0
is_late                       0.0
dtype: float64

In [172]:
model_df.shape

(95128, 24)

In [173]:
model_df.isnull().mean().sort_values(ascending=False).head(10)

customer_state          0.0
seller_state            0.0
price                   0.0
freight_value           0.0
payment_value           0.0
payment_installments    0.0
product_weight_g        0.0
product_length_cm       0.0
product_height_cm       0.0
product_width_cm        0.0
dtype: float64

In [174]:
#Split data
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=['is_late'])
y = model_df['is_late']


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [175]:
y_train.value_counts(normalize=True)

is_late
0    0.91899
1    0.08101
Name: proportion, dtype: float64

In [176]:
y_test.value_counts(normalize=True)

is_late
0    0.919006
1    0.080994
Name: proportion, dtype: float64

## Modeling Approach

We compare multiple models:

1. Logistic Regression → baseline (linear model)
2. Random Forest → captures non-linear relationships
3. XGBoost → optimized gradient boosting model

We focus on ROC-AUC and precision-recall trade-offs.

In [184]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import pandas as pd

In [185]:
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss"
    )
}

In [178]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['price', 'freight_value',
                                                   'payment_value',
                                                   'payment_installments',
                                                   'product_weight_g',
                                                   'product_length_cm',
                                                   'product_height_cm',
                                                   'product_width_cm',
                                                   'product_photos_qty',
                                                   'approval_time_hours',
                                                   'fr...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=300, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [186]:
comparison_results = []

for model_name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train)
    
    y_proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = pipe.predict(X_test)
    
    comparison_results.append({
        "Model": model_name,
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "Precision_Late": precision_score(y_test, y_pred),
        "Recall_Late": recall_score(y_test, y_pred),
        "F1_Late": f1_score(y_test, y_pred)
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df

,Model,ROC-AUC,Precision_Late,Recall_Late,F1_Late
0,Logistic Regression,0.743920,0.154140,0.680078,0.251319
1,Random Forest,0.778641,0.242917,0.523037,0.331756
2,XGBoost,0.785819,0.221327,0.606100,0.324249


In [187]:
final_threshold = 0.70

best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", models["XGBoost"])
])

best_model.fit(X_train, y_train)

y_proba_final = best_model.predict_proba(X_test)[:, 1]
y_pred_final = (y_proba_final >= final_threshold).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_proba_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall:", recall_score(y_test, y_pred_final))
print("F1:", f1_score(y_test, y_pred_final))

ROC-AUC: 0.7858193089209496
Precision: 0.33136094674556216
Recall: 0.36340038935756
F1: 0.34664190653048593


## Results

Final model: XGBoost

- ROC-AUC: ~0.78
- Precision (late): ~0.33 (at threshold 0.70)
- Recall (late): ~0.36

The model is better at ranking risk than making perfect classifications.

## Decision Threshold

Instead of using default threshold (0.5), we analyze multiple thresholds.

We selected ~0.70 to:
- Reduce false positives
- Maintain reasonable recall

This reflects a business decision rather than a purely technical one.

In [180]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

thresholds = np.arange(0.1, 0.9, 0.05)

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    precision = precision_score(y_test, y_pred_t)
    recall = recall_score(y_test, y_pred_t)
    print(f"Threshold: {t:.2f} | Precision: {precision:.3f} | Recall: {recall:.3f}")

Threshold: 0.10 | Precision: 0.089 | Recall: 0.991
Threshold: 0.15 | Precision: 0.097 | Recall: 0.970
Threshold: 0.20 | Precision: 0.108 | Recall: 0.929
Threshold: 0.25 | Precision: 0.122 | Recall: 0.885
Threshold: 0.30 | Precision: 0.137 | Recall: 0.833
Threshold: 0.35 | Precision: 0.156 | Recall: 0.787
Threshold: 0.40 | Precision: 0.178 | Recall: 0.739
Threshold: 0.45 | Precision: 0.199 | Recall: 0.679
Threshold: 0.50 | Precision: 0.221 | Recall: 0.622
Threshold: 0.55 | Precision: 0.245 | Recall: 0.562
Threshold: 0.60 | Precision: 0.271 | Recall: 0.498
Threshold: 0.65 | Precision: 0.297 | Recall: 0.425
Threshold: 0.70 | Precision: 0.334 | Recall: 0.360
Threshold: 0.75 | Precision: 0.367 | Recall: 0.282
Threshold: 0.80 | Precision: 0.427 | Recall: 0.209
Threshold: 0.85 | Precision: 0.500 | Recall: 0.122


In [181]:
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = pipeline.named_steps['model'].feature_importances_

import pandas as pd

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

feat_imp.head(15)

,feature,importance
46,cat__customer_state_SP,0.060963
13,num__purchase_month,0.042784
25,cat__customer_state_BA,0.038513
38,cat__customer_state_PR,0.038469
31,cat__customer_state_MG,0.038035
39,cat__customer_state_RJ,0.036538
14,num__same_state,0.035233
19,num__is_peak_month,0.031512
54,cat__seller_state_MA,0.030108
18,num__approval_to_estimated_days,0.029480


In [182]:
result_df = X_test.copy()
result_df['actual_is_late'] = y_test.values
result_df['late_probability'] = y_proba
result_df['predicted_is_late'] = y_pred_final

result_df.head()

,customer_state,seller_state,price,freight_value,payment_value,payment_installments,product_weight_g,product_length_cm,product_height_cm,product_width_cm,...,same_state,estimated_delivery_days,seller_order_count,item_count,approval_to_estimated_days,is_peak_month,delivery_pressure,actual_is_late,late_probability,predicted_is_late
87048,SP,SP,79.00,13.57,92.57,1,6900.0,28.0,55.0,37.0,...,1,18,1054,1,18,0,14.179431,0,0.172981,0
77085,RJ,RS,49.99,15.79,65.78,1,150.0,16.0,16.0,16.0,...,0,19,168,1,17,0,0.558477,0,0.472998,0
35696,SP,SP,29.90,8.11,38.01,1,250.0,16.0,5.0,15.0,...,1,14,1287,1,14,0,8.680675,0,0.108618,0
43766,SP,SP,59.00,7.78,66.78,1,150.0,16.0,2.0,11.0,...,1,16,1819,1,16,0,13.829532,0,0.162237,0
48825,SP,SP,399.00,9.66,408.66,8,9550.0,33.0,44.0,33.0,...,1,17,130,1,16,0,0.635534,0,0.200122,0


In [183]:
result_df.sort_values(
    by='late_probability',
    ascending=False
).head(20)

,customer_state,seller_state,price,freight_value,payment_value,payment_installments,product_weight_g,product_length_cm,product_height_cm,product_width_cm,...,same_state,estimated_delivery_days,seller_order_count,item_count,approval_to_estimated_days,is_peak_month,delivery_pressure,actual_is_late,late_probability,predicted_is_late
93233,SP,SP,159.90,9.28,169.18,2,317.0,18.0,10.0,12.0,...,1,2,19,1,2,0,0.134424,0,0.974485,1
20277,SP,SP,150.00,17.32,167.32,1,5450.0,23.0,24.0,23.0,...,1,4,20,1,2,0,0.139593,1,0.972620,1
57261,SP,SP,79.99,8.72,88.71,1,350.0,17.0,4.0,13.0,...,1,4,82,1,2,0,0.099706,1,0.963527,1
66473,BA,MA,64.99,37.14,102.13,3,350.0,19.0,12.0,13.0,...,0,22,389,1,22,0,2.769812,1,0.960012,1
87705,SP,SP,82.49,8.74,91.23,1,330.0,20.0,15.0,16.0,...,1,5,66,1,3,0,0.084369,1,0.954260,1
59867,SP,SP,147.99,12.06,160.05,1,1367.0,39.0,15.0,29.0,...,1,2,3,1,1,0,0.113238,0,0.951014,1
70152,SP,SP,42.89,13.60,56.49,1,742.0,37.0,32.0,29.0,...,1,5,97,1,3,0,0.127201,1,0.949564,1
81048,CE,SP,27.90,25.63,53.53,2,300.0,17.0,10.0,15.0,...,0,25,398,1,24,0,1.004442,1,0.947681,1
73019,DF,MA,64.99,22.95,87.94,5,350.0,19.0,12.0,13.0,...,0,20,389,1,20,0,16.230839,0,0.946552,1
54770,RJ,SP,149.94,43.25,193.19,1,13350.0,57.0,56.0,22.0,...,0,23,972,1,20,1,0.398644,1,0.944714,1


## Key Insights

- Delivery delays are strongly influenced by seasonality
- Tight delivery windows increase delay risk
- Logistic complexity (freight, product size) matters
- Model performance is limited by available data

Most importantly:
Improving features had more impact than changing models initially.

## Limitations

- No geographic distance data
- No carrier-level information
- Limited seller behavioral features

These factors likely limit model performance.

## Conclusion

This project demonstrates that:

- Predictive models are useful for ranking risk
- Perfect accuracy is not necessary for decision-making
- Threshold selection is critical in real-world applications